In [1]:
import pandas as pd
from catboost import CatBoostClassifier

In [3]:
train_path = 'datasets/train.csv'
test_path = 'datasets/test.csv'

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

print('train: ', train_df.shape)
print('test: ', test_df.shape)
print(train_df['Heart Disease'].value_counts())

train:  (630000, 15)
test:  (270000, 14)
Heart Disease
Absence     347546
Presence    282454
Name: count, dtype: int64


### Подготовка признаков

In [4]:
target_col = 'Heart Disease'
y = train_df[target_col].map({"Absence": 0, "Presence": 1})
X = train_df.drop(columns=[target_col])
X_test = test_df.copy()

In [5]:
cat_cols = [
    "Sex", "Chest pain type",
    "FBS over 120", "EKG results",
    "Exercise angina", "Slope of ST",
    "Number of vessels fluro", "Thallium",
]

In [7]:
# задаю индексы категориальных колонок для catboost
cat_features = [X.columns.get_loc(c) for c in cat_cols]
print("X:", X.shape, "y:", y.shape, "X_test:", X_test.shape)
print("cat feature idx:", cat_features)

X: (630000, 14) y: (630000,) X_test: (270000, 14)
cat feature idx: [2, 3, 6, 7, 9, 11, 12, 13]


### Разбиваю данные на выборки

In [8]:
from sklearn.model_selection import train_test_split
X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(X_train.shape, X_val.shape)

(504000, 14) (126000, 14)


### Обучаю модель catboost

In [9]:
from catboost import CatBoostClassifier
from sklearn.metrics import (
    f1_score, fbeta_score,
    roc_auc_score, average_precision_score
)

In [10]:
model = CatBoostClassifier(
    iterations=500, depth=6, learning_rate=0.1,
    loss_function='Logloss', eval_metric='AUC',
    random_seed=42, verbose=100
)

In [11]:
model.fit(
    X_train, y_train,
    cat_features=cat_features,
    eval_set=(X_val, y_val),
    use_best_model=True
)

0:	test: 0.9359527	best: 0.9359527 (0)	total: 164ms	remaining: 1m 21s
100:	test: 0.9551592	best: 0.9551592 (100)	total: 7.57s	remaining: 29.9s
200:	test: 0.9557438	best: 0.9557438 (200)	total: 15.5s	remaining: 23.1s
300:	test: 0.9559442	best: 0.9559442 (300)	total: 24.5s	remaining: 16.2s
400:	test: 0.9560690	best: 0.9560706 (395)	total: 33.1s	remaining: 8.17s
499:	test: 0.9561093	best: 0.9561094 (491)	total: 42.1s	remaining: 0us

bestTest = 0.9561094325
bestIteration = 491

Shrink model to first 492 iterations.


In [12]:
val_proba = model.predict_proba(X_val)[:, 1]
val_pred = (val_proba >= 0.5).astype(int)

In [14]:
metrics = {
    "f1": f1_score(y_val, val_pred),
    "f2": fbeta_score(y_val, val_pred, beta=2),
    "roc_auc": roc_auc_score(y_val, val_proba),
    "pr_auc": average_precision_score(y_val, val_proba),
}

In [15]:
metrics_df = pd.DataFrame([metrics])
print(metrics_df)

         f1       f2   roc_auc    pr_auc
0  0.876283  0.87157  0.956109  0.949587


#### Делаем сабмишн

In [16]:
test_proba = model.predict_proba(X_test)[:, 1]

submission = pd.DataFrame({
    "id": X_test["id"],
    "Heart Disease": test_proba
})

submission.to_csv("submissions/baseline.csv", index=False)
print(submission.head())

       id  Heart Disease
0  630000       0.944433
1  630001       0.006150
2  630002       0.985711
3  630003       0.002806
4  630004       0.172681
